In [1]:
# Importing ChexFound Code Base

import os
import sys
import importlib

# 1. Setup repository path
repo_dir = '/kaggle/working/CheXFound'
if not os.path.exists(os.path.join(repo_dir, 'chexfound', '__init__.py')):
    print("Cloning CheXFound codebase...")
    !rm -rf /kaggle/working/CheXFound
    !git clone https://github.com/SuyogMagar/CheXFound.git /kaggle/working/CheXFound

# 2. Register path permanently for this notebook session
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

importlib.invalidate_caches()

# 3. Test import to guarantee future cells will not fail
from chexfound.models import vision_transformer as vits
print("\n✓ SUCCESS: CheXFound codebase is imported, verified, and locked in sys.path.")

Cloning CheXFound codebase...
Cloning into '/kaggle/working/CheXFound'...
remote: Enumerating objects: 460, done.
remote: Counting objects: 100% (460/460), done.
remote: Compressing objects: 100% (358/358), done.
remote: Total 460 (delta 134), reused 407 (delta 96), pack-reused 0 (from 0)
Receiving objects: 100% (460/460), 24.14 MiB | 31.85 MiB/s, done.
Resolving deltas: 100% (134/134), done.

✓ SUCCESS: CheXFound codebase is imported, verified, and locked in sys.path.


/kaggle/working/CheXFound/chexfound/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/kaggle/working/CheXFound/chexfound/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/kaggle/working/CheXFound/chexfound/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [5]:
# Dataset Verification & Inspection: Montgomery County TB

import os
import glob
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Locate Montgomery metadata CSV
possible_csv = glob.glob('/kaggle/input/**/montgomery_metadata.csv', recursive=True)
assert len(possible_csv) > 0, "Could not find montgomery_metadata.csv in /kaggle/input/"
csv_path = possible_csv[0]
print(f"Found Metadata CSV: {csv_path}")

df = pd.read_csv(csv_path)

# 2. Map study_id to full image path across input folders
all_images = {os.path.basename(p): p for p in glob.glob('/kaggle/input/**/*.png', recursive=True)}

# Ensure .png extension is attached to match filenames
df['filename'] = df['study_id'].apply(lambda x: str(x) if str(x).endswith('.png') else f"{x}.png")
df['full_path'] = df['filename'].map(all_images)
df = df.dropna(subset=['full_path']).reset_index(drop=True)

# 3. Target label: Tuberculosis (0 = Normal, 1 = Tuberculosis)
# Montgomery encoding: 'normal' findings -> 0; filename ending in '_1.png' or abnormal findings -> 1
df['Tuberculosis'] = df['findings'].apply(
    lambda x: 0 if 'normal' in str(x).lower() else 1
)

# 4. Paper Section IV-B split (70:10:20 -> 96 Train, 14 Val, 28 Test)
indices = np.arange(len(df))
train_val_idx, test_idx = train_test_split(
    indices, test_size=28, random_state=42, stratify=df['Tuberculosis']
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=14, random_state=42, stratify=df.loc[train_val_idx, 'Tuberculosis']
)

df['split'] = 'train'
df.loc[val_idx, 'split'] = 'val'
df.loc[test_idx, 'split'] = 'test'

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df = df[df['split'] == 'val'].reset_index(drop=True)
test_df = df[df['split'] == 'test'].reset_index(drop=True)

print(f"\n✓ Verified Montgomery Scans: {len(df)}")
print(f"✓ Normal: {(df['Tuberculosis'] == 0).sum()} | Tuberculosis: {(df['Tuberculosis'] == 1).sum()}")
print(f"✓ Train Pool: {len(train_df)} | Val Pool: {len(val_df)} | Test Pool: {len(test_df)}")

# Display head of processed Montgomery dataset
df[['study_id', 'findings', 'Tuberculosis', 'split', 'full_path']].head(5)

Found Metadata CSV: /kaggle/input/datasets/raddar/tuberculosis-chest-xrays-montgomery/montgomery_metadata.csv

✓ Verified Montgomery Scans: 138
✓ Normal: 81 | Tuberculosis: 57
✓ Train Pool: 96 | Val Pool: 14 | Test Pool: 28


,study_id,findings,Tuberculosis,split,full_path
0,MCUCXR_0001_0.png,normal,0,train,/kaggle/input/datasets/raddar/tuberculosis-che...
1,MCUCXR_0002_0.png,normal,0,test,/kaggle/input/datasets/raddar/tuberculosis-che...
2,MCUCXR_0003_0.png,normal,0,test,/kaggle/input/datasets/raddar/tuberculosis-che...
3,MCUCXR_0004_0.png,normal,0,train,/kaggle/input/datasets/raddar/tuberculosis-che...
4,MCUCXR_0005_0.png,normal,0,val,/kaggle/input/datasets/raddar/tuberculosis-che...


In [3]:
# Load and Freeze the ChexFound Trained Parameters

import os
import glob
import torch
import torch.nn as nn
from chexfound.models import vision_transformer as vits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device for inference: {device}")

# 1. Locate teacher checkpoint
teacher_files = glob.glob('/kaggle/input/**/teacher_checkpoint.pth', recursive=True)
assert len(teacher_files) > 0, "Could not find teacher_checkpoint.pth in /kaggle/input/"
teacher_path = teacher_files[0]
print(f"Loading weights from: {teacher_path}")

# 2. Instantiate ViT-Large backbone (512x512 resolution, patch_size=16)
model = vits.vit_large(patch_size=16, img_size=512)

# 3. Load checkpoint and align state dict keys
teacher_ckpt = torch.load(teacher_path, map_location='cpu')
state_dict = teacher_ckpt.get('teacher', teacher_ckpt)
clean_state_dict = {
    (k[9:] if k.startswith('backbone.') else k): v 
    for k, v in state_dict.items()
}

missing_keys, unexpected_keys = model.load_state_dict(clean_state_dict, strict=False)

# 4. Freeze backbone and mount on GPU
model.eval()
model.to(device)
for param in model.parameters():
    param.requires_grad = False

print(f"\n✓ CheXFound ViT-Large Loaded:")
print(f"  - Missing keys: {len(missing_keys)}")
print(f"  - Unexpected keys: {len(unexpected_keys)}")
print(f"  - All parameters frozen: {all(not p.requires_grad for p in model.parameters())}")

# 5. Sanity check: Run a dummy forward pass through the last 4 blocks
dummy = torch.randn(1, 3, 512, 512).to(device)
with torch.no_grad():
    sample_layers = model.get_intermediate_layers(dummy, n=4, return_class_token=True)
    # Each block outputs: (patch_tokens [B, 1024, 1024], cls_token [B, 1024])
    patch_sample, cls_sample = sample_layers[-1]
    
print(f"\n✓ Forward check successful:")
print(f"  - Patch tokens shape: {patch_sample.shape} (1024 spatial patches)")
print(f"  - CLS token shape   : {cls_sample.shape} (1024 dimensions)")

Device for inference: cuda
Loading weights from: /kaggle/input/datasets/suyogmagar08/chexfound-weights/teacher_checkpoint.pth

✓ CheXFound ViT-Large Loaded:
  - Missing keys: 240
  - Unexpected keys: 305
  - All parameters frozen: True

✓ Forward check successful:
  - Patch tokens shape: torch.Size([1, 1024, 1024]) (1024 spatial patches)
  - CLS token shape   : torch.Size([1, 1024]) (1024 dimensions)


In [4]:
import os
import re
import sys
import glob
import subprocess
import importlib

# 0. Dependencies
for mod, pkg in [("open_clip", "open_clip_torch"), ("skmultilearn", "scikit-multilearn")]:
    try:
        __import__(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# 1. Setup repository
repo_dir = '/kaggle/working/CheXFound'
if not os.path.exists(os.path.join(repo_dir, 'chexfound', '__init__.py')):
    !rm -rf /kaggle/working/CheXFound
    !git clone https://github.com/RPIDIAL/CheXFound.git /kaggle/working/CheXFound || git clone https://github.com/SuyogMagar/CheXFound.git /kaggle/working/CheXFound

if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

importlib.invalidate_caches()
from chexfound.models import vision_transformer as vits

# 2. Check Cache or Extract Features for Montgomery
montgomery_cache = '/kaggle/working/montgomery_verified_tensors.pt'

if os.path.exists(montgomery_cache):
    print(f"✓ Found existing Montgomery cache at {montgomery_cache}. Loading...")
    bundle = torch.load(montgomery_cache, map_location='cpu')
    full_layers = bundle['full_layers']
    y_all = bundle['y_all']
    print(f"✓ Loaded {len(full_layers)} blocks, {len(y_all)} labels.")
else:
    print("Loading verified CheXFound backbone for Montgomery extraction...")
    teacher_files = glob.glob('/kaggle/input/**/teacher_checkpoint.pth', recursive=True)
    assert len(teacher_files) > 0, "teacher_checkpoint.pth missing!"
    teacher_ckpt = torch.load(teacher_files[0], map_location='cpu')
    raw_state = teacher_ckpt.get('teacher', teacher_ckpt)

    clean_state = {}
    for k, v in raw_state.items():
        if k.startswith('dino_head.') or k.startswith('ibot_head.'):
            continue
        new_k = k
        for pfx in ['module.', 'backbone.']:
            if new_k.startswith(pfx):
                new_k = new_k[len(pfx):]
        new_k = re.sub(r'^blocks\.\d+\.(\d+)\.', r'blocks.\1.', new_k)
        clean_state[new_k] = v

    has_swiglu = any('w12' in k for k in clean_state.keys())
    has_layerscale = any('ls1' in k for k in clean_state.keys())
    has_registers = 'register_tokens' in clean_state
    num_registers = clean_state['register_tokens'].shape[1] if has_registers else 0
    pos_shape = clean_state['pos_embed'].shape
    native_res = int(((pos_shape[1] - 1) ** 0.5) * 16)

    model = vits.vit_large(
        patch_size=16,
        img_size=native_res,
        init_values=1e-5 if has_layerscale else None,
        ffn_layer="swiglu" if has_swiglu else "mlp",
        block_chunks=0,
        num_register_tokens=num_registers
    )
    msg = model.load_state_dict(clean_state, strict=False)
    print(f"✓ Backbone Loaded -> Missing keys: {len(msg.missing_keys)} | Unexpected keys: {len(msg.unexpected_keys)}")
    assert len(msg.missing_keys) == 0, "Backbone weight loading failed!"

    model.eval().to(device)
    for p in model.parameters():
        p.requires_grad = False

    # 3. Locate Montgomery Scans
    search_patterns = [
        '/kaggle/input/**/MCUCXR_*.png',
        '/kaggle/input/**/*MCUCXR*.png',
        '/kaggle/input/**/*.png'
    ]
    montgomery_images = []
    for sp in search_patterns:
        found = glob.glob(sp, recursive=True)
        montgomery_images = sorted([p for p in found if 'MCUCXR' in os.path.basename(p)])
        if len(montgomery_images) > 0:
            break

    assert len(montgomery_images) > 0, "Montgomery images not found! Ensure the Montgomery dataset is attached."

    records = []
    for p in montgomery_images:
        base = os.path.splitext(os.path.basename(p))[0]
        label = 1 if base.endswith('_1') else 0
        records.append({'path': p, 'label': label})
    df_all = pd.DataFrame(records)
    y_all = torch.tensor(df_all['label'].values)
    print(f"Total Montgomery Scans Indexed: {len(df_all)} (Normal: {(df_all['label']==0).sum()}, TB: {(df_all['label']==1).sum()})")

    eval_transform = T.Compose([
        T.Resize((512, 512), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    class GenericDataset(Dataset):
        def __init__(self, df, transform):
            self.df = df
            self.transform = transform
        def __len__(self):
            return len(self.df)
        def __getitem__(self, idx):
            img = Image.open(self.df.iloc[idx]['path']).convert('RGB')
            return self.transform(img), self.df.iloc[idx]['label']

    loader = DataLoader(GenericDataset(df_all, eval_transform), batch_size=16, shuffle=False, num_workers=2)

    batch_layers = []
    with torch.no_grad():
        for imgs, _ in tqdm(loader, desc="Extracting Montgomery Tokens"):
            imgs = imgs.to(device)
            layers = model.get_intermediate_layers(imgs, n=4, return_class_token=True)
            batch_layers.append([(p.cpu(), c.cpu()) for p, c in layers])

    full_layers = []
    for b in range(4):
        p_b = torch.cat([batch[b][0] for batch in batch_layers], dim=0)
        c_b = torch.cat([batch[b][1] for batch in batch_layers], dim=0)
        full_layers.append((p_b, c_b))

    torch.save({'full_layers': full_layers, 'y_all': y_all}, montgomery_cache)
    print(f"✓ Cached Montgomery representations to {montgomery_cache}")

# 4. Paper Section IV-B Split: 70% Train (96), 10% Val (14), 20% Test (28)
y_np = y_all.numpy()
indices = np.arange(len(y_np))
train_val_idx, test_idx = train_test_split(indices, test_size=28, random_state=42, stratify=y_np)
train_idx, val_idx = train_test_split(train_val_idx, test_size=14, random_state=42, stratify=y_np[train_val_idx])

print(f"\nSplit Sizes -> Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

# =========================================================================
# PART A: Linear Probe Benchmark
# =========================================================================
X_cls = torch.cat([full_layers[b][1] for b in range(4)], dim=-1)
X_cls_norm = F.normalize(X_cls, p=2, dim=-1).numpy()

X_tr, y_tr = X_cls_norm[train_idx], y_np[train_idx]
X_va, y_va = X_cls_norm[val_idx], y_np[val_idx]
X_te, y_te = X_cls_norm[test_idx], y_np[test_idx]

best_c, best_val_auc = 1.0, -1.0
for c_val in [0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 20.0]:
    clf = LogisticRegression(C=c_val, max_iter=1000, solver='lbfgs')
    clf.fit(X_tr, y_tr)
    v_auc = roc_auc_score(y_va, clf.predict_proba(X_va)[:, 1])
    if v_auc > best_val_auc:
        best_val_auc = v_auc
        best_c = c_val

X_tr_val = np.vstack([X_tr, X_va])
y_tr_val = np.concatenate([y_tr, y_va])
lp_clf = LogisticRegression(C=best_c, max_iter=1000, solver='lbfgs')
lp_clf.fit(X_tr_val, y_tr_val)
lp_test_probs = lp_clf.predict_proba(X_te)[:, 1]

lp_auroc = roc_auc_score(y_te, lp_test_probs)
lp_auprc = average_precision_score(y_te, lp_test_probs)

print("\n" + "="*65)
print("     MONTGOMERY BENCHMARK: LINEAR PROBE")
print("="*65)
print(f"Linear Probe Test AUROC (28 scans) : {lp_auroc:.4f}  (Table IV Target: 0.990 [0.952 - 1.0])")
print(f"Linear Probe Test AUPRC (28 scans) : {lp_auprc:.4f}  (Table IV Target: 0.988 [0.939 - 1.0])")
print("="*65)

# =========================================================================
# PART B: GLoRI Head Benchmark
# =========================================================================
class PyramidPatchMerging(nn.Module):
    def __init__(self, embed_dim=1024, out_dim=768):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(embed_dim, out_dim),
            nn.ReLU(),
            nn.LayerNorm(out_dim)
        )
    def forward(self, patch_tokens):
        B, N, D = patch_tokens.shape
        H = W = int(N ** 0.5)
        x = patch_tokens.transpose(1, 2).view(B, D, H, W)
        p8 = F.adaptive_avg_pool2d(x, (4, 4)).flatten(2).transpose(1, 2)
        p4 = F.adaptive_avg_pool2d(x, (8, 8)).flatten(2).transpose(1, 2)
        p2 = F.adaptive_avg_pool2d(x, (16, 16)).flatten(2).transpose(1, 2)
        return self.proj(torch.cat([p8, p4, p2], dim=1))

class GLoRIHead(nn.Module):
    def __init__(self, in_dim=1024, glori_dim=768):
        super().__init__()
        self.disease_query = nn.Parameter(torch.randn(1, 1, glori_dim) * 0.02)
        self.patch_proj = nn.Sequential(nn.Linear(in_dim, glori_dim), nn.ReLU())
        self.temp_mlp = nn.Sequential(
            nn.Linear(glori_dim, glori_dim // 2),
            nn.Tanh(),
            nn.Linear(glori_dim // 2, 1)
        )
        self.mha_fine = nn.MultiheadAttention(embed_dim=glori_dim, num_heads=8, batch_first=True)
        self.pyramid = PyramidPatchMerging(embed_dim=in_dim, out_dim=glori_dim)
        self.mha_coarse = nn.MultiheadAttention(embed_dim=glori_dim, num_heads=8, batch_first=True)
        self.cls_proj = nn.Sequential(nn.Linear(in_dim, glori_dim), nn.LayerNorm(glori_dim))
        self.norm = nn.LayerNorm(glori_dim * 3)
        self.classifier = nn.Linear(glori_dim * 3, 1)

    def forward(self, layers_data):
        patch_tokens, cls_token = layers_data[-1]
        B = patch_tokens.shape[0]
        queries = self.disease_query.expand(B, -1, -1)
        proj_patches = self.patch_proj(patch_tokens)
        
        pooled = proj_patches.mean(dim=1)
        tau = torch.exp(self.temp_mlp(pooled)).unsqueeze(-1)
        queries_scaled = queries / (tau + 1e-6)
        fine_feat, _ = self.mha_fine(queries_scaled, proj_patches, proj_patches)
        
        coarse_patches = self.pyramid(patch_tokens)
        coarse_feat, _ = self.mha_coarse(queries, coarse_patches, coarse_patches)
        
        global_feat = self.cls_proj(cls_token).unsqueeze(1)
        integrated = torch.cat([fine_feat, coarse_feat, global_feat], dim=-1).squeeze(1)
        return self.classifier(self.norm(integrated))

def train_eval_glori(tr_idx, ev_idx, lr, epochs=80, batch_size=16):
    torch.manual_seed(42)
    m = GLoRIHead().to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    crit = nn.BCEWithLogitsLoss()
    
    n_tr = len(tr_idx)
    for epoch in range(epochs):
        m.train()
        perm = torch.randperm(n_tr)
        for i in range(0, n_tr, batch_size):
            b_idx = tr_idx[perm[i:i + batch_size]]
            b_layers = [(p[b_idx].to(device), c[b_idx].to(device)) for p, c in full_layers]
            b_y = y_all[b_idx].float().unsqueeze(1).to(device)
            opt.zero_grad()
            crit(m(b_layers), b_y).backward()
            opt.step()
        sched.step()
        
    m.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(ev_idx), batch_size):
            b_idx = ev_idx[i:i + batch_size]
            b_layers = [(p[b_idx].to(device), c[b_idx].to(device)) for p, c in full_layers]
            preds.append(np.atleast_1d(torch.sigmoid(m(b_layers)).cpu().squeeze(-1).numpy()))
    return np.concatenate(preds)

best_glori_lr, best_v_auc = 1e-4, -1.0
for lr in [5e-5, 1e-4, 2e-4, 5e-4]:
    v_preds = train_eval_glori(train_idx, val_idx, lr=lr, epochs=60, batch_size=16)
    v_auc = roc_auc_score(y_np[val_idx], v_preds)
    if v_auc > best_v_auc:
        best_v_auc = v_auc
        best_glori_lr = lr

glori_test_probs = train_eval_glori(train_val_idx, test_idx, lr=best_glori_lr, epochs=80, batch_size=16)
g_auroc = roc_auc_score(y_np[test_idx], glori_test_probs)
g_auprc = average_precision_score(y_np[test_idx], glori_test_probs)

print("\n" + "="*65)
print("     MONTGOMERY BENCHMARK: GLoRI HEAD")
print("="*65)
print(f"GLoRI Test AUROC (28 scans) : {g_auroc:.4f}  (Table IV Target: 1.000 [1.0 - 1.0])")
print(f"GLoRI Test AUPRC (28 scans) : {g_auprc:.4f}  (Table IV Target: 1.000 [1.0 - 1.0])")
print("="*65)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00
Device: cuda
Loading verified CheXFound backbone for Montgomery extraction...
✓ Backbone Loaded -> Missing keys: 0 | Unexpected keys: 0
Total Montgomery Scans Indexed: 138 (Normal: 80, TB: 58)


Extracting Montgomery Tokens: 100%|██████████| 9/9 [00:56<00:00,  6.31s/it]


✓ Cached Montgomery representations to /kaggle/working/montgomery_verified_tensors.pt

Split Sizes -> Train: 96 | Val: 14 | Test: 28

     MONTGOMERY BENCHMARK: LINEAR PROBE
Linear Probe Test AUROC (28 scans) : 0.9844  (Table IV Target: 0.990 [0.952 - 1.0])
Linear Probe Test AUPRC (28 scans) : 0.9791  (Table IV Target: 0.988 [0.939 - 1.0])

     MONTGOMERY BENCHMARK: GLoRI HEAD
GLoRI Test AUROC (28 scans) : 1.0000  (Table IV Target: 1.000 [1.0 - 1.0])
GLoRI Test AUPRC (28 scans) : 1.0000  (Table IV Target: 1.000 [1.0 - 1.0])
